# Retrieval-Augmented Generation with FAISS Semantic Search

This notebook demonstrates an example RAG flow using the FAISS vector store and semantic search capabilities shown in the previous notebook. The steps are as follows:

1. Load the persisted FAISS vector store
2. Run semantic retrieval over the archived corpus
3. Build a prompt from the top search results
4. Send the context to an LLM for a grounded answer

FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta. It is designed for efficient, high-dimensional similarity search and clustering of dense vectors.

### Learning Objectives

- How the FAISS-backed retriever is initialized
- What the raw retrieval results look like
- How search results are converted into LLM context
- How to produce an example RAG answer with either Groq or a local Ollama model


>__Note:__
>
>1. This notebook uses the vector store created in the exp-05_VectorStore_and_SemanticSearch notebook, to run the notebook in colab, you will need to download the `faiss` directory from the previous notebook to the same place (i.e., `sample_data/vector_store/faiss`) in this notebook's colab instance.
>2. This notebook assumes you have set up your environment variables for the LLM provider you intend to use (e.g., `GROQ_API_KEY` for Groq, or `OLLAMA_HOST` for Ollama). Make sure to configure these before running the notebook.
>   - For Groq, you can get your free API key from [Groq's website](https://groq.com/). 
>   - For Ollama, you can set up a local instance by following the instructions on [Ollama's documentation](https://ollama.com/docs/installation).

In [ ]:
# Colab bootstrap: install the required packages in this notebook
%pip install --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ wa-nlnz-toolkit
%pip install --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ wa-nlnz-semanticsearch

# additional package for LLM interaction
%pip install openai

In [ ]:
# Configure environment based on execution context
try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Set output directory based on execution environment
res_folder = "/content" if IN_COLAB else "./"

print(f"Running in Colab: {IN_COLAB}")
print(f"Results will be saved to: {res_folder}")

In [ ]:
# Download the vector store file from S3 if it doesn't exist locally
import os
import wa_nlnz_toolkit as want


os.makedirs(os.path.join(res_folder, "sample_data/vector_store/default"), exist_ok=True)
os.makedirs(os.path.join(res_folder, "sample_data/vector_store/faiss"), exist_ok=True)

bucket_name = "ndha-public-data-ap-southeast-2"
uri_prefix = "IIPC2026WAC/sample-data/vector_store/"
uri_data_all = want.list_s3_files(bucket_name, uri_prefix)
for uri_data in uri_data_all:
    want.download_s3_file(
        bucket_name,
        uri_data,
        os.path.join(
            res_folder, uri_data.replace("IIPC2026WAC/sample-data", "sample_data")
        ),
    )

In [ ]:
from pathlib import Path
import os
import sys
from typing import Dict, List
from google.colab import userdata

VECTOR_STORE_PATH = os.path.join(res_folder, "sample_data/vector_store/faiss")

print(f"Vector store path: {VECTOR_STORE_PATH}")
# For Colab
print(f"API Key Groq: {userdata.get('API_KEY_GROQ')}")
# For local environment
# print(f"API Key Groq: {os.environ.get('API_KEY_GROQ')}")

In [ ]:
from openai import OpenAI
from wa_nlnz_semanticsearch import SemanticSearchFAISS

## Step 1: Initialize the Retriever

This retrieval component loads a persisted FAISS index and the Hugging Face embedding model used for semantic search.

In [ ]:
search_engine = SemanticSearchFAISS(
    vector_store_path=str(VECTOR_STORE_PATH),
    embedding_model="all-MiniLM-L6-v2",
)

search_engine.get_stats()

## Step 2: Retrieve Relevant Chunks

The following cell posts a query to the retriever and inspects the returned results. Each result contains a text chunk, its source URL, and metadata such as the original document title and crawl date.

In [ ]:
query = "What was New Zealand's traffic light system for COVID-19?"
top_k = 5

results = search_engine.search(query, top_k=top_k)
print(f"Retrieved {len(results)} results for: {query}")

for index, result in enumerate(results, start=1):
    metadata = result.get("metadata", {})
    # preview the first 100 characters of the text chunk, replacing newlines with spaces
    preview = result.get("text", "")[:100].replace("\n", " ")
    print(f"\nResult #{index}")
    print(f"  score: {result.get('score')}")
    print(f"  url: {metadata.get('url', 'N/A')}")
    print(f"  snapshot_id: {metadata.get('snapshot_id', 'N/A')}")
    print(f"  page_idx: {metadata.get('page_idx', 'N/A')}")
    print(f"  text preview: {preview}...")

## Step 3: Convert Retrieval Results into RAG Context

The following helper function builds the RAG context. It takes the top results, truncates each passage, and concatenates them into a prompt. This is the context that will be sent to the LLM.

In [ ]:
def build_rag_context(
    results: List[Dict], max_results: int = 5, max_chars: int = 500
) -> str:
    context_parts = []
    for index, result in enumerate(results[:max_results], start=1):
        text = result.get("text", "")[:max_chars]
        url = result.get("metadata", {}).get("url", "N/A")
        snapshot_id = result.get("metadata", {}).get("snapshot_id", "N/A")
        context_parts.append(
            f"Source {index} | snapshot={snapshot_id} | {url}:\n{text}"
        )
    return "\n\n".join(context_parts)


context = build_rag_context(results, max_chars=5000)
print(context)

## Step 4: Build the RAG Prompt

This prompt asks the model to answer the query directly, synthesize across sources, and keep the response grounded in the retrieved archive snippets.

In [ ]:
def build_rag_prompt(query: str, context: str) -> str:
    return f"""Based on the following search results about '{query}', provide a comprehensive summary that answers the query.

Search Results:
---------------
{context}
---------------
Please provide a clear, concise summary that:
1. Directly answers the query
2. Synthesizes information from multiple sources
3. Highlights key facts and dates
4. Makes uncertainty explicit if the retrieved evidence is incomplete

Summary:"""


prompt = build_rag_prompt(query, context)
print(prompt)

## Step 5: Configure an LLM Client

The app supports two backends:

- Groq via the OpenAI-compatible API
- Ollama running locally via its OpenAI-compatible endpoint

Set one of these before running the next cell:

- `export GROQ_API_KEY=...` to use Groq
- start Ollama locally and pull a model such as `llama3.1:8b`

In [ ]:
RAG_PROVIDER = os.environ.get("RAG_PROVIDER", "groq").lower()
RAG_MODEL = os.environ.get(
    "RAG_MODEL",
    "openai/gpt-oss-20b" if RAG_PROVIDER == "groq" else "llama3.1:8b",
)


def create_llm_client(provider: str) -> OpenAI:
    if provider == "groq":
        # For Colab
        api_key = userdata.get('API_KEY_GROQ')
        # For local environment
        # api_key = os.environ.get("API_KEY_GROQ")
        # or set the key directly
        # api_key = "put_your_groq_api_key_here"
        if not api_key:
            raise ValueError(
                "Set API_KEY_GROQ in your environment before using the Groq backend."
            )
        return OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

    if provider == "ollama":
        return OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

    raise ValueError(f"Unsupported RAG_PROVIDER: {provider}")


print(f"Provider: {RAG_PROVIDER}")
print(f"Model: {RAG_MODEL}")

## Step 6: Run the End-to-End RAG Flow

The function below mirrors the behavior of the web app: retrieve top chunks, build the context, create the prompt, and request a grounded answer from the configured LLM.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that summarizes information from web archives "
    "about COVID-19 policies and guidelines. Use only the provided context, and say "
    "when the evidence is incomplete."
)


def answer_with_rag(
    query: str,
    top_k: int = 5,
    provider: str = RAG_PROVIDER,
    model: str = RAG_MODEL,
) -> Dict[str, object]:
    retrieved = search_engine.search(query, top_k=top_k)
    context = build_rag_context(retrieved, max_results=min(top_k, 5))
    prompt = build_rag_prompt(query, context)

    client = create_llm_client(provider)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        max_tokens=4098,
    )

    answer = response.choices[0].message.content
    return {
        "query": query,
        "model": model,
        "provider": provider,
        "sources_used": len(retrieved[:5]),
        "answer": answer,
        "results": retrieved,
        "context": context,
    }

Feel free to experiment with the query. Keep in mind it is querying the covid19.govt.nz dataset provided by the National Library of New Zealand. Harvests of the website can be viewed here https://ndhadeliver.natlib.govt.nz/webarchive/*/https://covid19.govt.nz/

In [ ]:
rag_response = answer_with_rag(
    query="Summarize the traffic light system and when it applied.",
    top_k=5,
)

print(f"Model: {rag_response['model']}")
print(f"Sources used: {rag_response['sources_used']}")
print()
print(rag_response["answer"])

## Step 7: Inspect the Evidence Used by the Answer

When evaluating a RAG pipeline, inspect both the generated answer and the retrieved evidence. That is the fastest way to tell whether a bad answer came from retrieval quality, prompt design, or model behavior.

In [ ]:
for index, result in enumerate(rag_response["results"], start=1):
    metadata = result.get("metadata", {})
    print(f"\nSource #{index}")
    print(f"  score: {result.get('score')}")
    print(f"  url: {metadata.get('url', 'N/A')}")
    print(f"  snapshot_id: {metadata.get('snapshot_id', 'N/A')}")
    print(f"  text: {result.get('text', '')[:500]}...")

### Citation

**Title:** Retrieval-Augmented Generation with FAISS Semantic Search (Jupyter notebook)  
**Author:** Yizhe Zhan, Ben O'Brien  
**Affiliation:** National Library of New Zealand, Wellington  
**Last updated:** 2026‑04‑14  

**Contact:** yizhe.git@gmail.com  
**Repository:** https://github.com/NLNZDigitalPreservation/wa-nlnz-toolkit